# Decision Tree Control Model For Colab

This notebook converts the tree training flow into a Colab notebook, but uses a **plain `DecisionTreeRegressor`** as the control model instead of XGBoost.

Estimated training time on the current dataset (`~4.19M` rows / `1,000` tickers):
- Full dataset, `5` CV folds + final fit: about `8-25 minutes`
- `max_tickers=200`: often `2-6 minutes`

A simple sklearn decision tree is CPU-bound, so Colab GPU does **not** speed this notebook up much.
The saved artifact goes to `pipeline/models` and is usable from `pipeline.predict` under model name `tree`.


In [ ]:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU
!pip -q install prophet xgboost torch scikit-learn pandas numpy joblib


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/diploma')  # Change if your repo is elsewhere in Colab or Drive.
DATASET_PATH = PROJECT_DIR / 'dataset' / 'stock_prices_20y.db'
MODELS_DIR = PROJECT_DIR / 'pipeline' / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'

for path in [MODELS_DIR, RESULTS_DIR, ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Missing dataset at {DATASET_PATH}. Update PROJECT_DIR or place the repo there first.'
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Project dir :', PROJECT_DIR)
print('Dataset path :', DATASET_PATH)


In [ ]:
import sqlite3
import time
from datetime import datetime

import numpy as np
import pandas as pd

with sqlite3.connect(DATASET_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()

print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')


In [ ]:
import json
import pickle
from dataclasses import asdict

from sklearn.tree import DecisionTreeRegressor

from src.tree.train_tree import (
    Config,
    FEATURE_COLS,
    build_grouped_time_folds,
    load_prices_from_sqlite,
    make_features,
    regression_metrics,
    split_train_test_per_ticker,
)

RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
TREE_PARAMS = {
    'max_depth': 6,
    'min_samples_leaf': 100,
    'min_samples_split': 500,
    'random_state': 42,
}

cfg = Config(
    db_path=str(DATASET_PATH),
    model_output_path=str(MODELS_DIR / f'{RUN_TAG}-decision_tree.pkl'),
    config_output_path=str(ARTIFACTS_DIR / f'{RUN_TAG}-decision_tree_config.json'),
)
print(cfg)
print('Decision tree params:', TREE_PARAMS)


In [ ]:
start_time = time.perf_counter()

print('Loading data...')
raw_df = load_prices_from_sqlite(cfg.db_path, cfg.table_name)
all_tickers = list(raw_df['ticker'].unique())
if cfg.max_tickers > 0 and len(all_tickers) > cfg.max_tickers:
    rng = np.random.default_rng(cfg.random_state)
    all_tickers = rng.choice(all_tickers, cfg.max_tickers, replace=False).tolist()
    raw_df = raw_df[raw_df['ticker'].isin(all_tickers)].reset_index(drop=True)
    print(f'Subsampled to {cfg.max_tickers} tickers')
else:
    print(f'Using all {len(all_tickers)} tickers')

feat_df = make_features(raw_df)
train_df, test_df = split_train_test_per_ticker(feat_df, cfg.test_size, cfg.target_horizon)
feature_cols = [c for c in FEATURE_COLS if c in train_df.columns]
train_clean = train_df[feature_cols + ['target', 'ticker', 'date', 'close']].dropna().copy()
test_clean = test_df[feature_cols + ['target', 'ticker', 'date', 'close']].dropna().copy()

X_train = train_clean[feature_cols].to_numpy(dtype=np.float32)
y_train = train_clean['target'].to_numpy(dtype=np.float32)
X_test = test_clean[feature_cols].to_numpy(dtype=np.float32)
y_test = test_clean['target'].to_numpy(dtype=np.float32)

print(f'Train rows: {len(X_train):,} | Test rows: {len(X_test):,}')
folds = build_grouped_time_folds(train_clean, cfg.n_splits, cfg.target_horizon)
print('CV folds:', len(folds))

cv_results = []
for fold_i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(f'\n=== Fold {fold_i}/{len(folds)} ===')
    model = DecisionTreeRegressor(**TREE_PARAMS)
    model.fit(X_train[tr_idx], y_train[tr_idx])
    metrics = regression_metrics(y_train[va_idx], model.predict(X_train[va_idx]))
    cv_results.append(metrics)
    print(metrics)

cv_df = pd.DataFrame(cv_results)
print('\nCV mean metrics')
print(cv_df.mean(numeric_only=True).to_string())

print('\nTraining final model...')
final_model = DecisionTreeRegressor(**TREE_PARAMS)
final_model.fit(X_train, y_train)
y_test_pred = final_model.predict(X_test)
test_metrics = regression_metrics(y_test, y_test_pred)
print('Held-out test metrics:', test_metrics)

config_dict = asdict(cfg)
config_dict['feature_cols'] = feature_cols
config_dict['model_family'] = 'decision_tree_regressor'
config_dict.update(TREE_PARAMS)

payload = {
    'model': final_model,
    'feature_cols': feature_cols,
    'config': config_dict,
    'cv_results': cv_results,
    'test_metrics': test_metrics,
}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
with open(cfg.config_output_path, 'w') as f:
    json.dump(config_dict, f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-decision_tree_predictions.csv'
out_df = test_clean[['ticker', 'date', 'close']].copy()
out_df['y_true'] = y_test
out_df['y_pred'] = y_test_pred
out_df.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
